# MMIRAG — leakage and image-overlap checks (no API key, no cost)

Two checks that need the original corpus files on Drive:

1. **Leakage into the article corpus of the frontier runs.** The same word 8-gram check as for the atlas, run on the Set 1 article PDFs. Set `ARTICLE_FOLDER` to the folder that holds them.
2. **Overlap between the 60 WebPath query images and the image corpus.** A perceptual-hash comparison (catches the same image resized or recompressed) and the PubMedCLIP similarity used by the pipeline (catches crops). Set `QUERY_IMAGES` and `IMAGE_CORPUS`. A GPU runtime makes this faster but is not required.

Both results are saved in `MMIRAG_R3` on Drive (`r3_check_article_corpus_leakage.csv`, `r3_check_image_overlap.csv`, `r3_checks_summary.txt`) and downloaded at the end. Nothing else is changed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pymupdf

## Check 1 — article corpus (about a minute)

In [ ]:
# ---- Check 1: leakage of the 250 primary questions into the ARTICLE corpus of the frontier runs (free, no API)
import os, re, glob, json
import pandas as pd, fitz
WORKDIR = '/content/drive/MyDrive/MMIRAG_R3'                 # clean250.csv is here
ARTICLE_FOLDER = "/content/drive/MyDrive/PDF's"              # <- folder with the Set 1 article PDFs of the frontier runs
pdfs = sorted(glob.glob(os.path.join(ARTICLE_FOLDER, '*.pdf')))
print(f'{len(pdfs)} PDFs in {ARTICLE_FOLDER}:'); [print('  ', os.path.basename(p)) for p in pdfs]
tok = lambda s: re.findall(r'[a-z0-9]+', s.lower())
def grams(w, n): return {tuple(w[i:i + n]) for i in range(len(w) - n + 1)}
G8, G5, per_file = set(), set(), {}
for p in pdfs:
    d = fitz.open(p); w = tok(' '.join(pg.get_text() for pg in d)); d.close()
    g8 = grams(w, 8); per_file[os.path.basename(p)] = g8; G8 |= g8; G5 |= grams(w, 5)
Q = pd.read_csv(f'{WORKDIR}/clean250.csv', dtype=str, keep_default_na=False)
rows = []
for _, r in Q.iterrows():
    w = tok(r.question); g8 = grams(w, 8); g5 = grams(w, 5)
    shared = g8 & G8
    rows.append(dict(id=r.id, n8=len(g8), shared8=len(shared), share8=len(shared) / max(len(g8), 1),
                     any5=bool(g5 & G5), files=';'.join(f for f, g in per_file.items() if shared & g),
                     example=' | '.join(' '.join(x) for x in list(shared)[:3])))
L = pd.DataFrame(rows).sort_values('share8', ascending=False)
L.to_csv(f'{WORKDIR}/r3_check_article_corpus_leakage.csv', index=False)
txt = (f'Article-corpus leakage check ({len(pdfs)} PDFs, {len(Q)} primary questions):\n'
       f'  questions sharing any word 8-gram: {(L.shared8 > 0).sum()}; maximum share {L.share8.max():.1%}\n'
       f'  questions sharing at least one word 5-gram: {L.any5.mean():.0%}\n')
print(txt); print(L.head(10)[['id', 'shared8', 'n8', 'share8', 'files', 'example']].to_string())
open(f'{WORKDIR}/r3_checks_summary.txt', 'a').write(txt)

## Check 2 — query images vs image corpus (a few minutes)

In [ ]:
# ---- Check 2: do any of the 60 query images also appear in the IMAGE corpus? (free, no API)
!pip -q install imagehash
import os, io, glob
import numpy as np, pandas as pd, fitz, imagehash, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
QUERY_IMAGES = '/content/drive/MyDrive/WebPath_images'        # <- folder with the 60 WebPath question images
IMAGE_CORPUS = '/content/drive/MyDrive/image_corpus_pdfs'     # <- folder with the PDF(s) the 1,396 corpus images were extracted from
q_files = sorted(f for f in glob.glob(os.path.join(QUERY_IMAGES, '**', '*'), recursive=True)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')))
print(f'{len(q_files)} query images in {QUERY_IMAGES}')
corpus = []                                                     # (pdf, page, PIL image)
for p in sorted(glob.glob(os.path.join(IMAGE_CORPUS, '*.pdf'))):
    d = fitz.open(p)
    for pno in range(len(d)):
        for im in d[pno].get_images(full=True):
            try:
                pix = fitz.Pixmap(d, im[0])
                if pix.width < 50 or pix.height < 50: continue
                if pix.n - pix.alpha >= 4: pix = fitz.Pixmap(fitz.csRGB, pix)
                corpus.append((os.path.basename(p), pno + 1, Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')))
            except Exception:
                pass
    d.close()
print(f'{len(corpus)} corpus images extracted (the paper reports 1,396)')
# perceptual hashes: robust to resizing and recompression
ch = [(imagehash.phash(c[2]), imagehash.dhash(c[2])) for c in corpus]
# PubMedCLIP cosine similarity (the encoder of the pipeline): robust to crops and colour changes
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
clip = CLIPModel.from_pretrained('flaviagiammarino/pubmed-clip-vit-base-patch32').to(dev).eval()
proc = CLIPProcessor.from_pretrained('flaviagiammarino/pubmed-clip-vit-base-patch32')
@torch.no_grad()
def emb(imgs):
    out = []
    for i in range(0, len(imgs), 64):
        x = proc(images=imgs[i:i + 64], return_tensors='pt').to(dev)
        e = clip.get_image_features(**x)
        if not torch.is_tensor(e):                     # newer transformers versions return an output object
            e = e.image_embeds if getattr(e, 'image_embeds', None) is not None else e.pooler_output
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu())
    return torch.cat(out).numpy()
CE = emb([c[2] for c in corpus])
rows = []
for f in q_files:
    im = Image.open(f).convert('RGB'); ph, dh = imagehash.phash(im), imagehash.dhash(im)
    dist = np.array([min(ph - a, dh - b) for a, b in ch]); j = int(dist.argmin())
    sim = CE @ emb([im])[0]; k = int(sim.argmax())
    rows.append(dict(query=os.path.relpath(f, QUERY_IMAGES), min_hash_distance=int(dist[j]), hash_match=f'{corpus[j][0]} p{corpus[j][1]}',
                     max_clip_cosine=float(sim[k]), clip_match=f'{corpus[k][0]} p{corpus[k][1]}'))
R = pd.DataFrame(rows).sort_values('min_hash_distance')
R.to_csv('/content/drive/MyDrive/MMIRAG_R3/r3_check_image_overlap.csv', index=False)
txt = (f'Image-overlap check ({len(q_files)} query images vs {len(corpus)} corpus images):\n'
       f'  likely duplicates (perceptual-hash distance <= 8): {(R.min_hash_distance <= 8).sum()}\n'
       f'  maximum PubMedCLIP cosine similarity: median {R.max_clip_cosine.median():.3f}, max {R.max_clip_cosine.max():.3f}; '
       f'>= 0.95 for {(R.max_clip_cosine >= 0.95).sum()} queries\n')
print(txt); print(R.head(15).to_string())
open('/content/drive/MyDrive/MMIRAG_R3/r3_checks_summary.txt', 'a').write(txt)
print('Open the matches with the smallest hash distance / highest cosine and look at them: only an identical figure counts as overlap.')

## Download the results

In [ ]:
import zipfile, os
from google.colab import files
W = '/content/drive/MyDrive/MMIRAG_R3'
zp = '/content/MMIRAG_R3_checks.zip'
with zipfile.ZipFile(zp, 'w') as z:
    for f in ('r3_check_article_corpus_leakage.csv', 'r3_check_image_overlap.csv', 'r3_checks_summary.txt'):
        if os.path.exists(f'{W}/{f}'): z.write(f'{W}/{f}', f)
print(open(f'{W}/r3_checks_summary.txt').read())
files.download(zp)